
# House Price Prediction 

## Libraries Used

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor

In [4]:
train_df = pd.read_csv(r"train.csv")
test_df = pd.read_csv(r"test.csv")

print("Train Shape:", train_df.shape)
print("Test Shape:", test_df.shape)

train_df.head()

Train Shape: (1460, 81)
Test Shape: (1459, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## Data Cleaning & Missing Values

In [6]:
missing = train_df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print(missing.head(20))


PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtExposure      38
BsmtFinType2      38
BsmtQual          37
BsmtCond          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64


In [7]:
train_df = train_df[train_df['GrLivArea'] < 4000]
train_df['SalePrice'] = np.log1p(train_df['SalePrice'])

## Feature Selection

In [9]:

X = train_df.drop(['SalePrice'], axis=1)
y = train_df['SalePrice']

# Save IDs
test_ids = test_df['Id']

# Separate feature types
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object', 'string']).columns

print("Numeric Features:", len(numeric_features))
print("Categorical Features:", len(categorical_features))


Numeric Features: 37
Categorical Features: 43


## Preprocessing Pipeline

In [10]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

## Train Validation Split

In [11]:

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_valid.shape)

(1164, 80) (292, 80)


## Model Training

In [12]:

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(
        n_estimators=300,
        max_depth=20,
        random_state=42
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        random_state=42
    ),
    'XGBoost': XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )
}

results = []

for name, model in models.items():

    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    pipeline.fit(X_train, y_train)

    preds = pipeline.predict(X_valid)

    rmse = np.sqrt(mean_squared_error(y_valid, preds))
    mae = mean_absolute_error(y_valid, preds)
    r2 = r2_score(y_valid, preds)

    results.append([name, rmse, mae, r2])

results_df = pd.DataFrame(results, columns=['Model', 'RMSE', 'MAE', 'R2'])
results_df.sort_values(by='RMSE')


,Model,RMSE,MAE,R2
3,XGBoost,0.128381,0.087862,0.895645
2,Gradient Boosting,0.128507,0.087284,0.895440
0,Linear Regression,0.129138,0.087687,0.894410
1,Random Forest,0.148265,0.102536,0.860815


## Best Model Training

In [15]:

best_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    ))
])

best_model.fit(X, y)

test_predictions = best_model.predict(test_df)


test_predictions = np.expm1(test_predictions)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': test_predictions
})

submission.to_csv('submission.csv', index=False)

print("Submission file created successfully!")
submission.head()


Submission file created successfully!


,Id,SalePrice
0,1461,122493.953125
1,1462,154190.046875
2,1463,184730.453125
3,1464,188534.984375
4,1465,179331.562500


## Cross Validation

In [16]:

cv_scores = cross_val_score(
    best_model,
    X,
    y,
    cv=5,
    scoring='neg_root_mean_squared_error'
)

rmse_scores = -cv_scores

print("Cross Validation RMSE Scores:")
print(rmse_scores)

print("\nAverage RMSE:", rmse_scores.mean())


Cross Validation RMSE Scores:
[0.10748442 0.11816315 0.12198221 0.10496053 0.11417699]

Average RMSE: 0.11335346039002743
